# LeetCode 705 — Design HashSet bằng Python

**Link đề:** https://leetcode.com/problems/design-hashset/
**Độ khó:** Easy · **Chủ đề:** Array, Hash Table, Linked List, Design, Hash Function

```python
class MyHashSet:

    def __init__(self):
        ...

    def add(self, key: int) -> None:
        ...

    def remove(self, key: int) -> None:
        ...

    def contains(self, key: int) -> bool:
        ...
```

## Mục tiêu học
1. Hiểu **bên trong** `set` của Python hoạt động thế nào: hàm băm (hash function), bucket, va chạm (collision).
2. Tự cài HashSet **không dùng** `set`/`dict` theo 3 cách: list tuần tự → mảng đánh dấu trực tiếp (direct addressing) → **băm + chaining**.
3. Biết chọn số bucket, vì sao hay dùng **số nguyên tố**, và load factor ảnh hưởng tốc độ thế nào.
4. Trace trạng thái các bucket sau mỗi thao tác, chứng minh invariant, phân tích Time/Space trung bình và xấu nhất.
5. Chuẩn bị cho bài 706 Design HashMap (cùng khung, thêm value).

## 1. Đề bài và ví dụ

Thiết kế HashSet **không dùng thư viện bảng băm có sẵn**. Cài class `MyHashSet`:
- `add(key)`: thêm `key` vào tập.
- `contains(key)`: trả `True` nếu `key` có trong tập.
- `remove(key)`: xóa `key`; nếu không có thì không làm gì.

**Ví dụ:**

| Lệnh | Tập sau lệnh | Trả về |
|---|---|---|
| `MyHashSet()` | `{}` | `null` |
| `add(1)` | `{1}` | `null` |
| `add(2)` | `{1, 2}` | `null` |
| `contains(1)` | `{1, 2}` | `true` |
| `contains(3)` | `{1, 2}` | `false` |
| `add(2)` | `{1, 2}` (không trùng) | `null` |
| `contains(2)` | `{1, 2}` | `true` |
| `remove(2)` | `{1}` | `null` |
| `contains(2)` | `{1}` | `false` |

**Constraints:** `0 <= key <= 10^6`; tối đa `10^4` lời gọi `add`, `remove`, `contains`.

In [1]:
OPS = ["MyHashSet", "add", "add", "contains", "contains", "add", "contains", "remove", "contains"]
ARGS = [[], [1], [2], [1], [3], [2], [2], [2], [2]]
EXP = [None, None, None, True, False, None, True, None, False]

def run_ops(cls, ops, args):
    out, obj = [], None
    for op, a in zip(ops, args):
        if op == ops[0]:
            obj = cls(*a)
            out.append(None)
        else:
            out.append(getattr(obj, op)(*a))
    return out

print("Kịch bản:", list(zip(OPS, ARGS)))

Kịch bản: [('MyHashSet', []), ('add', [1]), ('add', [2]), ('contains', [1]), ('contains', [3]), ('add', [2]), ('contains', [2]), ('remove', [2]), ('contains', [2])]


`run_ops` mô phỏng cách LeetCode chạy test: lệnh đầu là tạo đối tượng, các lệnh sau gọi phương thức theo tên (`getattr(obj, "add")` lấy phương thức `add`).

## 2. Nền tảng: `set` của Python làm được gì?

Để biết mục tiêu, xem `set` có sẵn (chỉ dùng **để so sánh**, không dùng trong lời giải):

In [2]:
s = set()
s.add(1); s.add(2); s.add(2)
print(s, 1 in s, 3 in s)
s.discard(2)          # discard: xóa nếu có, không lỗi nếu không có
s.discard(99)
print(s)

{1, 2} True False
{1}


Ba thao tác đều **trung bình `O(1)`**. Bí mật nằm ở **hàm băm**: biến key thành chỉ số của một mảng.

### 2.1. Mảng (list) truy cập theo chỉ số là `O(1)`

`arr[i]` đọc/ghi ngay vị trí `i` mà không duyệt. Nếu key là số nhỏ, ta có thể dùng chính key làm chỉ số!

In [3]:
arr = [False] * 10          # list 10 phần tử, đều False
arr[3] = True
print(arr)
print("3 có trong tập?", arr[3], "| 4 có?", arr[4])

[False, False, False, True, False, False, False, False, False, False]
3 có trong tập? True | 4 có? False


`[False] * 10` tạo list gồm 10 phần tử `False` (nhân list = lặp lại).

### 2.2. Phép chia lấy dư `%` — hàm băm đơn giản nhất

`key % M` luôn nằm trong `0..M-1`. Dùng nó để "gấp" key lớn vào một mảng nhỏ `M` ô (gọi là **bucket**).

In [4]:
M = 7
for key in [1, 2, 8, 15, 1000000, 10]:
    print(f"key = {key:>7} -> bucket {key % M}")

key =       1 -> bucket 1
key =       2 -> bucket 2
key =       8 -> bucket 1
key =      15 -> bucket 1
key = 1000000 -> bucket 1
key =      10 -> bucket 3


Thấy `1`, `8`, `15` đều vào bucket `1` → **va chạm (collision)**: các key khác nhau cùng một bucket. Phải có cách chứa nhiều key trong một bucket.

### 2.3. List lồng nhau: list của các list

Mỗi bucket là một list nhỏ. **Cẩn thận**: `[[]] * M` tạo `M` tham chiếu tới **cùng một** list! Phải dùng list comprehension.

In [5]:
wrong = [[]] * 3
wrong[0].append(5)
print("sai  :", wrong)          # cả 3 bucket đều có 5!

right = [[] for _ in range(3)]
right[0].append(5)
print("đúng :", right)

sai  : [[5], [5], [5]]
đúng : [[5], [], []]


## 3. Cách 1 — List tuần tự (baseline)

Lưu mọi key vào một list. `contains` và `remove` phải **duyệt cả list**.

In [6]:
class MyHashSetList:
    def __init__(self):
        self.data = []
    def add(self, key):
        if not self.contains(key):        # tránh trùng
            self.data.append(key)
    def remove(self, key):
        if key in self.data:              # 'in' trên list: duyệt O(n)
            self.data.remove(key)
    def contains(self, key):
        for x in self.data:
            if x == key:
                return True
        return False

print(run_ops(MyHashSetList, OPS, ARGS))
print(run_ops(MyHashSetList, OPS, ARGS) == EXP)

[None, None, None, True, False, None, True, None, False]
True


**Nút thắt:** mỗi thao tác `O(n)` với `n` = số key hiện có. `10^4` lời gọi × tới `10^4` phần tử → khoảng `10^8` phép so sánh trường hợp xấu. Chậm, và không phải bảng băm.

## 4. Cách 2 — Mảng đánh dấu trực tiếp (direct addressing)

Vì `0 <= key <= 10^6`, tạo list `present` dài `10^6 + 1`; `present[key] = True` nghĩa là key có trong tập.

In [7]:
class MyHashSetDirect:
    def __init__(self):
        self.present = [False] * (10**6 + 1)
    def add(self, key):
        self.present[key] = True
    def remove(self, key):
        self.present[key] = False
    def contains(self, key):
        return self.present[key]

print(run_ops(MyHashSetDirect, OPS, ARGS) == EXP)

True


- Ưu: mọi thao tác **`O(1)` thực sự** (không chỉ trung bình), code cực ngắn.
- Nhược: luôn tốn bộ nhớ `10^6 + 1` ô dù chỉ lưu vài key; không dùng được nếu key lớn (ví dụ `10^18`) hoặc là chuỗi.

Biến thể tiết kiệm: dùng `bytearray` (mỗi ô 1 byte) thay list (mỗi ô là con trỏ 8 byte).

In [8]:
import sys
lst = [False] * (10**6 + 1)
ba = bytearray(10**6 + 1)
print("list     :", sys.getsizeof(lst) // 1024, "KB")
print("bytearray:", sys.getsizeof(ba) // 1024, "KB")

list     : 7812 KB
bytearray: 976 KB


## 5. Insight — băm + chaining (separate chaining)

Kết hợp hai ý:
1. Một mảng **nhỏ** `M` bucket; `h(key) = key % M` chọn bucket → bộ nhớ cỡ `M + n` thay vì `10^6`.
2. Mỗi bucket là một **list nhỏ** chứa các key va chạm. Nếu key phân bố đều, mỗi bucket chỉ có khoảng `n / M` key (gọi là **load factor** `α`).

→ mỗi thao tác chỉ duyệt **một bucket** ≈ `O(1 + α)`.

Chọn `M` là **số nguyên tố** (ví dụ `769`, `1009`, `10007`) để giảm va chạm có quy luật: nếu key có chung ước với `M` (ví dụ key toàn số chẵn và `M = 1000`), chúng dồn vào ít bucket.

In [9]:
keys = list(range(0, 20000, 10))          # 2000 key, đều chia hết cho 10
for M in [1000, 1009]:
    buckets_used = len({k % M for k in keys})   # dùng set CHỈ để thống kê minh họa
    print(f"M = {M}: số bucket được dùng = {buckets_used}, bucket dài nhất ≈ {max([sum(1 for k in keys if k % M == r) for r in range(M)])}")

M = 1000: số bucket được dùng = 100, bucket dài nhất ≈ 20


M = 1009: số bucket được dùng = 1009, bucket dài nhất ≈ 2


Với `M = 1000` (không nguyên tố, chung ước 10 với key) chỉ 100 bucket được dùng, mỗi bucket dài 20. Với `M = 1009` (nguyên tố), key trải đều hơn.

## 6. Cách 3 — Băm + chaining bằng list (lời giải chính)

Mổ xẻ từng phần:
- `__init__`: `self.M = 1009`, `self.buckets = [[] for _ in range(self.M)]`.
- `_bucket(key)`: trả về list bucket của key: `self.buckets[key % self.M]`. Dấu `_` đầu tên: quy ước "hàm nội bộ".
- `add`: lấy bucket; nếu key **chưa** có thì `append`.
- `remove`: lấy bucket; nếu key có thì `remove`.
- `contains`: `key in bucket` (duyệt list ngắn).

In [10]:
class MyHashSetChain:
    def __init__(self):
        self.M = 1009
        self.buckets = [[] for _ in range(self.M)]

    def _bucket(self, key):
        return self.buckets[key % self.M]

    def add(self, key):
        bucket = self._bucket(key)
        if key not in bucket:
            bucket.append(key)

    def remove(self, key):
        bucket = self._bucket(key)
        if key in bucket:
            bucket.remove(key)

    def contains(self, key):
        return key in self._bucket(key)

print(run_ops(MyHashSetChain, OPS, ARGS))
print(run_ops(MyHashSetChain, OPS, ARGS) == EXP)

[None, None, None, True, False, None, True, None, False]
True


Chú ý: `key in bucket` với `bucket` là **list** — đây là duyệt tuần tự trên list nhỏ, **không** dùng bảng băm có sẵn, nên hợp lệ với đề.

`bucket = self._bucket(key)` trả về **chính** list trong `self.buckets` (tham chiếu), nên `bucket.append(...)` sửa trực tiếp bảng.

## 7. TRACE — nhìn thấy bucket thay đổi

Dùng `M = 5` (nhỏ để dễ nhìn) và các key cố ý va chạm: `1, 6, 11` đều `% 5 = 1`.

| Lệnh | bucket = key % 5 | Hành động | Các bucket khác rỗng |
|---|:---:|---|---|
| add(1) | 1 | thêm | {1: [1]} |
| add(6) | 1 | thêm (va chạm) | {1: [1, 6]} |
| add(3) | 3 | thêm | {1: [1, 6], 3: [3]} |
| add(6) | 1 | đã có → bỏ qua | không đổi |
| contains(11) | 1 | duyệt [1, 6] → không | False |
| remove(1) | 1 | xóa 1 | {1: [6], 3: [3]} |
| contains(6) | 1 | duyệt [6] → có | True |

In [11]:
class MyHashSetTrace(MyHashSetChain):
    def __init__(self, M=5):
        self.M = M
        self.buckets = [[] for _ in range(M)]
    def state(self):
        return {i: b for i, b in enumerate(self.buckets) if b}
    def add(self, key):
        bucket = self._bucket(key)
        action = "đã có -> bỏ qua" if key in bucket else "thêm"
        if key not in bucket:
            bucket.append(key)
        print(f"add({key:>2})      bucket {key % self.M}: {action:16} | {self.state()}")
    def remove(self, key):
        bucket = self._bucket(key)
        action = "xóa" if key in bucket else "không có -> bỏ qua"
        if key in bucket:
            bucket.remove(key)
        print(f"remove({key:>2})   bucket {key % self.M}: {action:16} | {self.state()}")
    def contains(self, key):
        bucket = self._bucket(key)
        res = key in bucket
        print(f"contains({key:>2}) bucket {key % self.M}: duyệt {bucket} -> {res}")
        return res

hs = MyHashSetTrace(5)
hs.add(1); hs.add(6); hs.add(3); hs.add(6)
hs.contains(11)
hs.remove(1)
hs.contains(6)
hs.remove(42)

add( 1)      bucket 1: thêm             | {1: [1]}
add( 6)      bucket 1: thêm             | {1: [1, 6]}
add( 3)      bucket 3: thêm             | {1: [1, 6], 3: [3]}
add( 6)      bucket 1: đã có -> bỏ qua  | {1: [1, 6], 3: [3]}
contains(11) bucket 1: duyệt [1, 6] -> False
remove( 1)   bucket 1: xóa              | {1: [6], 3: [3]}
contains( 6) bucket 1: duyệt [6] -> True
remove(42)   bucket 2: không có -> bỏ qua | {1: [6], 3: [3]}


Trace kịch bản mẫu của đề:

In [12]:
_ = run_ops(lambda: MyHashSetTrace(5), OPS, ARGS)

add( 1)      bucket 1: thêm             | {1: [1]}
add( 2)      bucket 2: thêm             | {1: [1], 2: [2]}
contains( 1) bucket 1: duyệt [1] -> True
contains( 3) bucket 3: duyệt [] -> False
add( 2)      bucket 2: đã có -> bỏ qua  | {1: [1], 2: [2]}
contains( 2) bucket 2: duyệt [2] -> True
remove( 2)   bucket 2: xóa              | {1: [1]}
contains( 2) bucket 2: duyệt [] -> False


## 8. Code nộp LeetCode

Bản chaining (đúng tinh thần "tự cài bảng băm"):

In [13]:
class MyHashSet:

    def __init__(self):
        self.size = 1009
        self.buckets = [[] for _ in range(self.size)]

    def _bucket(self, key: int) -> list:
        return self.buckets[key % self.size]

    def add(self, key: int) -> None:
        bucket = self._bucket(key)
        if key not in bucket:
            bucket.append(key)

    def remove(self, key: int) -> None:
        bucket = self._bucket(key)
        if key in bucket:
            bucket.remove(key)

    def contains(self, key: int) -> bool:
        return key in self._bucket(key)

print(run_ops(MyHashSet, OPS, ARGS) == EXP)

True


Bản mảng đánh dấu (ngắn nhất, `O(1)` chắc chắn, tốn ~1 MB với `bytearray`):

```python
class MyHashSet:
    def __init__(self):
        self.present = bytearray(10**6 + 1)
    def add(self, key: int) -> None:
        self.present[key] = 1
    def remove(self, key: int) -> None:
        self.present[key] = 0
    def contains(self, key: int) -> bool:
        return self.present[key] == 1
```

## 9. Cách 4 — Chaining bằng danh sách liên kết (linked list)

Trong sách giáo khoa, mỗi bucket thường là **linked list**: mỗi node giữ `key` và con trỏ `next`. Ta dùng **node giả (dummy/sentinel)** ở đầu mỗi bucket để thao tác xóa không cần xử lý riêng node đầu.

In [14]:
class Node:
    def __init__(self, key, nxt=None):
        self.key = key
        self.next = nxt

class MyHashSetLinked:
    def __init__(self, size=1009):
        self.size = size
        self.heads = [Node(-1) for _ in range(size)]   # dummy đầu mỗi bucket

    def _find_prev(self, key):
        prev = self.heads[key % self.size]
        while prev.next is not None and prev.next.key != key:
            prev = prev.next
        return prev            # prev.next là node chứa key, hoặc None

    def add(self, key):
        prev = self._find_prev(key)
        if prev.next is None:
            prev.next = Node(key)

    def remove(self, key):
        prev = self._find_prev(key)
        if prev.next is not None:
            prev.next = prev.next.next     # "nối tắt" bỏ node chứa key

    def contains(self, key):
        return self._find_prev(key).next is not None

print(run_ops(MyHashSetLinked, OPS, ARGS) == EXP)

True


`_find_prev` trả về node **đứng trước** node chứa key — để xóa ta cần node trước đó (`prev.next = prev.next.next`). Dummy node đảm bảo luôn có "node trước", kể cả khi key ở đầu bucket.

## 10. Vì sao đúng?

**Invariant:** *Một key `k` đang có trong tập ⇔ `k` xuất hiện **đúng một lần** trong bucket `k % M`, và không xuất hiện ở bucket nào khác.*

- Ban đầu mọi bucket rỗng, tập rỗng ✔.
- `add(k)`: chỉ chạm bucket `k % M`; chỉ `append` khi `k` chưa có → vẫn đúng một lần ✔.
- `remove(k)`: nếu `k` có (đúng một lần) thì `list.remove` xóa lần xuất hiện đó → không còn ✔; nếu không có thì không đổi ✔.
- `contains(k)`: theo invariant, chỉ cần tìm trong bucket `k % M` ✔.

Điều kiện "**đúng một lần**" quan trọng: nếu `add` cho phép trùng, `list.remove` chỉ xóa **một** bản và `contains` sẽ sai sau khi remove (xem lỗi 2).

## 11. Độ phức tạp và so sánh

Gọi `n` = số key đang lưu, `M` = số bucket, `U = 10^6 + 1` = miền giá trị key, `α = n / M` (load factor).

| Cách | add / remove / contains | Bộ nhớ | Ghi chú |
|---|---|---|---|
| 1. List tuần tự | `O(n)` | `O(n)` | Baseline |
| 2. Mảng đánh dấu | `O(1)` | `O(U)` | Chỉ khi miền key nhỏ |
| 3. Băm + chaining (list) | `O(1 + α)` trung bình, `O(n)` xấu nhất | `O(M + n)` | **Nên nộp** |
| 4. Băm + linked list | như 3 | `O(M + n)` | Cách kinh điển |

Ở đây `n ≤ 10^4`, `M = 1009` → `α ≤ 10`: mỗi bucket trung bình ≤ 10 phần tử. Muốn nhanh hơn có thể lấy `M = 10007`. Xấu nhất là khi mọi key va chạm vào cùng bucket (ví dụ toàn bội số của `M`).

In [15]:
import random, time
rng = random.Random(705)
ops = [(rng.choice(["add", "remove", "contains"]), rng.randint(0, 10**6)) for _ in range(10**4)]
for cls in [MyHashSetList, MyHashSetDirect, MyHashSet, MyHashSetLinked]:
    obj = cls()
    t0 = time.perf_counter()
    for op, k in ops:
        getattr(obj, op)(k)
    print(f"{cls.__name__:16} 10^4 thao tác: {time.perf_counter() - t0:.4f}s")

MyHashSetList    10^4 thao tác: 0.2723s
MyHashSetDirect  10^4 thao tác: 0.0054s
MyHashSet        10^4 thao tác: 0.0046s
MyHashSetLinked  10^4 thao tác: 0.0067s


In [16]:
hs = MyHashSet()
for k in rng.sample(range(10**6 + 1), 10**4):
    hs.add(k)
lengths = [len(bk) for bk in hs.buckets]
print("số bucket:", len(lengths), "| bucket dài nhất:", max(lengths), "| trung bình:", round(sum(lengths) / len(lengths), 2))

số bucket: 1009 | bucket dài nhất: 20 | trung bình: 9.91


## 12. Bộ test — mẫu, biên, phản ví dụ, đối chiếu

In [17]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:36} PASS")

check("mẫu đề", run_ops(MyHashSet, OPS, ARGS), EXP)
ops2 = ["MyHashSet", "add", "add", "add", "remove", "contains", "add", "contains"]
args2 = [[], [1], [10001], [1], [1], [1], [1010], [1]]
check("va chạm 1 & 1010 (cùng % 1009)", run_ops(MyHashSet, ops2, args2), [None, None, None, None, None, False, None, False])
s = MyHashSet(); s.add(0); s.add(10**6)
check("biên key = 0 và 10^6", (s.contains(0), s.contains(10**6), s.contains(1)), (True, True, False))
s.remove(5)
check("remove key không tồn tại", s.contains(0), True)
s.add(7); s.add(7); s.remove(7)
check("add trùng rồi remove 1 lần", s.contains(7), False)

mẫu đề                               PASS
va chạm 1 & 1010 (cùng % 1009)       PASS
biên key = 0 và 10^6                 PASS
remove key không tồn tại             PASS
add trùng rồi remove 1 lần           PASS


In [18]:
impls = [MyHashSetList, MyHashSetDirect, MyHashSetChain, MyHashSet, MyHashSetLinked]
for trial in range(200):
    objs = [c() for c in impls]
    oracle = set()          # set có sẵn CHỈ dùng làm đáp án chuẩn khi kiểm thử
    pool = [rng.randint(0, 10**6) for _ in range(5)] + [1009 * i for i in range(5)]
    for _ in range(100):
        op, k = rng.choice(["add", "remove", "contains"]), rng.choice(pool)
        if op == "add":
            oracle.add(k)
        elif op == "remove":
            oracle.discard(k)
        exp = (k in oracle) if op == "contains" else None
        for o in objs:
            assert getattr(o, op)(k) == exp
print("200 kịch bản ngẫu nhiên (có cố ý va chạm): 5 cài đặt khớp set chuẩn")

200 kịch bản ngẫu nhiên (có cố ý va chạm): 5 cài đặt khớp set chuẩn


## 13. Lỗi người mới hay gặp

### Lỗi 1 — `[[]] * M` tạo các bucket dùng chung

In [19]:
class WrongShared(MyHashSetChain):
    def __init__(self):
        self.M = 1009
        self.buckets = [[]] * self.M          # SAI

w = WrongShared()
w.add(5)
w.add(1010)
print("bucket 0:", w.buckets[0], "| bucket 7:", w.buckets[7], "| bucket 1008:", w.buckets[1008])
print("contains(5):", w.contains(5), "| contains(6):", w.contains(6))

bucket 0: [5, 1010] | bucket 7: [5, 1010] | bucket 1008: [5, 1010]
contains(5): True | contains(6): False


Vì sao sai: mọi bucket là **cùng một** list → mọi key dồn vào một chỗ (bucket 0, 7, 1008 đều thấy `[5, 1010]`). Kết quả `contains` tình cờ vẫn đúng vì nó duyệt cả list chung, nhưng mỗi thao tác thành `O(n)` — mất hẳn ý nghĩa băm; sang bài 706 (HashMap) lỗi này còn dễ gây sai giá trị. Sửa: `[[] for _ in range(M)]`.

In [20]:
print("số list khác nhau trong WrongShared:", len({id(bk) for bk in w.buckets}))
print("số list khác nhau trong MyHashSet  :", len({id(bk) for bk in MyHashSet().buckets}))

số list khác nhau trong WrongShared: 1
số list khác nhau trong MyHashSet  : 1009


### Lỗi 2 — `add` không kiểm tra trùng

In [21]:
class WrongDup(MyHashSetChain):
    def add(self, key):
        self._bucket(key).append(key)         # SAI: cho phép trùng

w = WrongDup()
w.add(2); w.add(2); w.remove(2)
print("sai: contains(2) sau khi remove =", w.contains(2), "| đúng: False")

sai: contains(2) sau khi remove = True | đúng: False


Vì sao sai: bucket thành `[2, 2]`; `list.remove` chỉ xóa bản đầu → còn một bản. Sửa: `if key not in bucket: append`.

### Lỗi 3 — `list.remove` khi key không có
`[1, 2].remove(5)` ném `ValueError`. Đề yêu cầu "không có thì không làm gì". Sửa: kiểm `if key in bucket` trước.

In [22]:
try:
    [1, 2].remove(5)
except ValueError as e:
    print("Lỗi:", e)

Lỗi: list.remove(x): x not in list


### Lỗi 4 — Dùng `set`/`dict` có sẵn
`self.s = set()` qua được test nhưng **vi phạm đề** ("without using any built-in hash table libraries"). Phỏng vấn sẽ coi như chưa giải.

### Lỗi 5 — Mảng đánh dấu thiếu 1 ô
`[False] * 10**6` chỉ có chỉ số `0..999999`; `add(10**6)` → `IndexError`. Phải là `10**6 + 1`.

## 14. Mẹo nhận dạng pattern và biến thể

**Khi nào tự cài bảng băm / dùng mảng đánh dấu:**
- Đề cấm thư viện, hoặc key là số nguyên trong miền nhỏ → **direct addressing** (mảng đếm/đánh dấu) rất mạnh: ví dụ đếm ký tự bằng list 26 ô.
- Hiểu chaining giúp giải thích "vì sao `dict` trung bình `O(1)` nhưng xấu nhất `O(n)`".

Python thật sự dùng **open addressing** (dò tìm ô trống kế tiếp) thay vì chaining, và tự **resize** khi load factor vượt ngưỡng.

### 14.1. Biến thể A — tự tăng kích thước (rehash) khi load factor > 0.75

In [23]:
class MyHashSetResize:
    def __init__(self):
        self.M, self.n = 8, 0
        self.buckets = [[] for _ in range(self.M)]
    def _rehash(self):
        old = self.buckets
        self.M *= 2
        self.buckets = [[] for _ in range(self.M)]
        for bk in old:
            for k in bk:
                self.buckets[k % self.M].append(k)
    def add(self, key):
        bk = self.buckets[key % self.M]
        if key not in bk:
            bk.append(key); self.n += 1
            if self.n / self.M > 0.75:
                self._rehash()
    def remove(self, key):
        bk = self.buckets[key % self.M]
        if key in bk:
            bk.remove(key); self.n -= 1
    def contains(self, key):
        return key in self.buckets[key % self.M]

r = MyHashSetResize()
for k in range(100):
    r.add(k * 37)
print("n =", r.n, "| M =", r.M, "| load =", round(r.n / r.M, 2), "| contains(37*50):", r.contains(37 * 50))

n = 100 | M = 256 | load = 0.39 | contains(37*50): True


### 14.2. Biến thể B — bitset: mỗi key 1 bit
Dùng số nguyên Python làm dãy bit: `mask |= 1 << key` để thêm, `mask >> key & 1` để kiểm tra. Bộ nhớ khoảng `10^6 / 8 ≈ 125 KB`.

In [24]:
mask = 0
mask |= 1 << 3
mask |= 1 << 10
print(bin(mask), (mask >> 3) & 1, (mask >> 4) & 1)
mask &= ~(1 << 3)        # xóa bit 3
print(bin(mask))

0b10000001000 1 0
0b10000000000


## 15. Bài tập tự luyện

**Bài 1 (tính tay).** Với `M = 7`, sau `add(3), add(10), add(17), add(4), remove(10)`, bucket 3 chứa gì? Gán list vào `ex1_bucket3`.

**Bài 2 (đoán output).** `a = [[]] * 2; a[1].append(9); print(a)` in gì? Gán vào `ex2_output` (dạng list).

**Bài 3 (TODO).** Tự viết `Ex3HashSet` bằng chaining (không xem code trên).

**Bài 4 (biến thể).** `Ex4CountingSet` cho phép trùng (multiset): `add`, `remove` (xóa **một** bản), `count(key)`. Không dùng `dict`.

**Bài 5 (biến thể).** `Ex5BitSet` dùng một số nguyên làm bitset với 3 phương thức như đề.

**Bài 6 (mini challenge).** `Ex6OpenAddressing`: HashSet dùng **open addressing / linear probing** với mảng cố định `M = 20011` ô; xóa bằng đánh dấu "tombstone".

In [25]:
ex1_bucket3 = None   # TODO
ex2_output = None    # TODO

class Ex3HashSet:
    def __init__(self):
        # TODO
        pass
    def add(self, key): pass        # TODO
    def remove(self, key): pass     # TODO
    def contains(self, key): pass   # TODO

# assert run_ops(Ex3HashSet, OPS, ARGS) == EXP

In [26]:
class Ex4CountingSet:
    def __init__(self): pass        # TODO
    def add(self, key): pass        # TODO
    def remove(self, key): pass     # TODO
    def count(self, key): pass      # TODO

class Ex5BitSet:
    def __init__(self): pass        # TODO
    def add(self, key): pass        # TODO
    def remove(self, key): pass     # TODO
    def contains(self, key): pass   # TODO

class Ex6OpenAddressing:
    def __init__(self): pass        # TODO
    def add(self, key): pass        # TODO
    def remove(self, key): pass     # TODO
    def contains(self, key): pass   # TODO

# c = Ex4CountingSet(); c.add(5); c.add(5); c.remove(5); assert c.count(5) == 1
# assert run_ops(Ex5BitSet, OPS, ARGS) == EXP
# assert run_ops(Ex6OpenAddressing, OPS, ARGS) == EXP

## 16. Gợi ý 3 mức

- **Mức 1:** Bucket = `key % M`; mọi thao tác chỉ đụng một bucket.
- **Mức 2:** Bài 4: mỗi bucket lưu các cặp `[key, count]`. Bài 6: mỗi ô là `None` (trống, chưa từng dùng), `TOMB` (đã xóa) hoặc một key; khi tìm, dò `i, i+1, ...` (vòng quanh `% M`) tới khi gặp key hoặc `None` — **không** dừng ở `TOMB`.
- **Mức 3:** Bài 6 `add`: nếu đã `contains` thì thôi; ngược lại dò từ `key % M` tới ô đầu tiên là `None` hoặc `TOMB` rồi ghi key.

## 17. Đáp án tham khảo

Bài 1: `3 % 7 = 3`, `10 % 7 = 3`, `17 % 7 = 3`, `4 % 7 = 4` → bucket 3 là `[3, 10, 17]`, sau `remove(10)` còn `[3, 17]`.
Bài 2: `[[9], [9]]` vì hai phần tử là cùng một list.

In [27]:
ans_ex1_bucket3 = [3, 17]
ans_ex2_output = [[9], [9]]

class AnsEx3HashSet:
    def __init__(self):
        self.M = 1009
        self.b = [[] for _ in range(self.M)]
    def add(self, key):
        bk = self.b[key % self.M]
        if key not in bk:
            bk.append(key)
    def remove(self, key):
        bk = self.b[key % self.M]
        if key in bk:
            bk.remove(key)
    def contains(self, key):
        return key in self.b[key % self.M]

class AnsEx4CountingSet:
    def __init__(self):
        self.M = 1009
        self.b = [[] for _ in range(self.M)]
    def _find(self, key):
        for pair in self.b[key % self.M]:
            if pair[0] == key:
                return pair
        return None
    def add(self, key):
        p = self._find(key)
        if p:
            p[1] += 1
        else:
            self.b[key % self.M].append([key, 1])
    def remove(self, key):
        p = self._find(key)
        if p:
            p[1] -= 1
            if p[1] == 0:
                self.b[key % self.M].remove(p)
    def count(self, key):
        p = self._find(key)
        return p[1] if p else 0

class AnsEx5BitSet:
    def __init__(self):
        self.mask = 0
    def add(self, key):
        self.mask |= 1 << key
    def remove(self, key):
        self.mask &= ~(1 << key)
    def contains(self, key):
        return (self.mask >> key) & 1 == 1

class AnsEx6OpenAddressing:
    TOMB = -1
    def __init__(self):
        self.M = 20011
        self.slots = [None] * self.M
    def _probe(self, key):
        i = key % self.M
        while self.slots[i] is not None:
            yield i
            i = (i + 1) % self.M
        yield i
    def contains(self, key):
        for i in self._probe(key):
            if self.slots[i] == key:
                return True
        return False
    def add(self, key):
        if self.contains(key):
            return
        i = key % self.M
        while self.slots[i] is not None and self.slots[i] != self.TOMB:
            i = (i + 1) % self.M
        self.slots[i] = key
    def remove(self, key):
        for i in self._probe(key):
            if self.slots[i] == key:
                self.slots[i] = self.TOMB
                return

h = AnsEx3HashSet()
for k in [3, 10, 17, 4]:
    h.add(k)
h.remove(10)
bk7 = [[] for _ in range(7)]
for k in [3, 10, 17, 4]:
    bk7[k % 7].append(k)
bk7[3].remove(10)
assert ans_ex1_bucket3 == bk7[3]
a = [[]] * 2; a[1].append(9)
assert ans_ex2_output == a
c = AnsEx4CountingSet(); c.add(5); c.add(5); c.remove(5)
assert c.count(5) == 1 and c.count(6) == 0
for cls in [AnsEx3HashSet, AnsEx5BitSet, AnsEx6OpenAddressing]:
    assert run_ops(cls, OPS, ARGS) == EXP
for trial in range(150):
    objs = [AnsEx3HashSet(), AnsEx5BitSet(), AnsEx6OpenAddressing()]
    oracle = set()
    pool = [rng.randint(0, 10**6) for _ in range(4)] + [20011 * i + 5 for i in range(4)]
    for _ in range(80):
        op, k = rng.choice(["add", "remove", "contains"]), rng.choice(pool)
        if op == "add": oracle.add(k)
        elif op == "remove": oracle.discard(k)
        exp = (k in oracle) if op == "contains" else None
        for o in objs:
            assert getattr(o, op)(k) == exp, (type(o).__name__, op, k)
print("Đáp án tham khảo: PASS toàn bộ")

Đáp án tham khảo: PASS toàn bộ


Giải thích `yield` trong `_probe`: hàm có `yield` là **generator** — mỗi lần vòng `for` cần giá trị kế tiếp, hàm chạy tới `yield` tiếp theo rồi tạm dừng. Nhờ đó ta duyệt chuỗi ô dò mà không tạo list.

## 18. Template nhớ nhanh và checklist

```python
class MyHashSet:
    def __init__(self):
        self.size = 1009                                   # số nguyên tố
        self.buckets = [[] for _ in range(self.size)]      # KHÔNG dùng [[]] * size
    def _bucket(self, key):
        return self.buckets[key % self.size]
    def add(self, key):
        b = self._bucket(key)
        if key not in b: b.append(key)
    def remove(self, key):
        b = self._bucket(key)
        if key in b: b.remove(key)
    def contains(self, key):
        return key in self._bucket(key)
```

Câu thần chú: **"Băm để chọn bucket, duyệt bucket nhỏ để tìm; không trùng khi thêm, có rồi mới xóa."**

### Checklist trước khi submit
- [ ] Không dùng `set`/`dict` trong lời giải.
- [ ] Khởi tạo bucket bằng list comprehension, không `[[]] * M`.
- [ ] `add` không thêm trùng; `remove` kiểm tra tồn tại trước.
- [ ] Mảng đánh dấu (nếu dùng) có `10**6 + 1` ô.
- [ ] Test key `0`, `10^6`, các key va chạm (`k` và `k + M`).
- [ ] Nói được: trung bình `O(1 + n/M)`, xấu nhất `O(n)`, bộ nhớ `O(M + n)`.